# ACE-Net — Evaluation Results

Reproduces the evaluation tables of ACE-Net (Yu et al., *Electronics* 2025,
14, 4420) for this replication:

- **Table 2** — speech–text (MDCNN + cross-attention) emotion recognition
- **Table 3** — facial (FV-LiteNet) emotion recognition
- **Table 4** — multimodal forgery detection by pairing type

All metrics are computed on held-out test partitions. Figures and CSV summaries
are written to `MyDrive/acenet_results/`.

## Evaluation protocol

- **Split.** 80% train / 10% validation / 10% test, seed 42.
  - CREMA-D: actor-disjoint partition — no speaker appears in more than one
    partition. The same partition is shared between Stage-1 and Stage-2, so
    actors evaluated in Stage-2 were excluded from Stage-1 extractor training.
  - MELD: stratified by emotion.
- **Stage-1** models are evaluated on the test partition of their dataset.
- **Stage-2** is evaluated on the test partition of a 1:1 genuine/forged set
  whose positives are split evenly between Emotion-Tampering (P1) and
  Cross-Identity Spliced (P2) forgeries.
- Inputs are per-sample z-scored log-Mel spectrograms cropped to a fixed length;
  keyframe weights are uniform. These controls remove loudness, duration, and
  pooling-pattern cues from the class signal.

## Metric definitions

- **Accuracy (ACC)** — correct predictions / total predictions.
- **Weighted-F1** — per-class F1 averaged with weights equal to class support.
- **Precision** — TP / (TP + FP).
- **Recall** — TP / (TP + FN).
- **F1** — harmonic mean of precision and recall.
- **AUC** — area under the ROC curve; a threshold-independent measure of
  ranking separability between classes.
- **Confusion matrix** — counts of (true class, predicted class); displayed
  row-normalized (each row sums to 1).

## 1. Environment

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git checkout feat/acenet-training-pipeline
!git log --oneline -1

In [ ]:
!pip -q install torch torchvision torchaudio transformers librosa pillow scikit-learn
import torch, sys
sys.path.insert(0, '/content/Baseline_Training')
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 2. Data and checkpoints

Mount Drive, then unzip **`acenet_data.zip`** (must contain `CREMA-D/` with
`GENUINE_LastHalf`, `GENUINE_FirstHalf`, `FAKE_Paradigm1`, `FAKE_Paradigm2`, and
`MELD/` with `train`,`dev`,`test`). Checkpoints are read from
`MyDrive/acenet_ckpts/` (the four Stage-1 `.pt` and `stage2_acenet.pt`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, glob, shutil

DRIVE_ZIP = '/content/drive/MyDrive/acenet_data.zip'   # adjust if needed
LOCAL_ZIP = '/content/acenet_data.zip'
assert os.path.exists(DRIVE_ZIP), f'data zip not found: {DRIVE_ZIP}'
shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
DST = '/content/Baseline_Training/data'; os.makedirs(DST, exist_ok=True)
with zipfile.ZipFile(LOCAL_ZIP) as z: z.extractall(DST)
for target in ['CREMA-D','MELD']:
    for h in [d for d in glob.glob(f'{DST}/**/{target}', recursive=True) if os.path.isdir(d)]:
        want=os.path.join(DST,target)
        if os.path.abspath(h)!=os.path.abspath(want): shutil.move(h, want)
for sub in ['GENUINE_LastHalf','GENUINE_FirstHalf','FAKE_Paradigm1','FAKE_Paradigm2']:
    loose=os.path.join(DST,sub)
    if os.path.isdir(loose):
        os.makedirs(os.path.join(DST,'CREMA-D'),exist_ok=True)
        shutil.move(loose, os.path.join(DST,'CREMA-D',sub))

# checkpoints
os.makedirs('checkpoints', exist_ok=True)
for f in glob.glob('/content/drive/MyDrive/acenet_ckpts/*.pt'):
    shutil.copy(f, f'checkpoints/{os.path.basename(f)}')
print('data/ ->', os.listdir(DST))
print('checkpoints ->', os.listdir('checkpoints'))

## 3. Verify layout

In [ ]:
import os
DST='/content/Baseline_Training/data'
req_dirs=['CREMA-D/GENUINE_LastHalf','CREMA-D/GENUINE_FirstHalf',
          'CREMA-D/FAKE_Paradigm1','CREMA-D/FAKE_Paradigm2','MELD/train']
missing=[p for p in req_dirs if not os.path.isdir(os.path.join(DST,p))]
assert not missing, f'missing data dirs: {missing}'
ck=set(os.listdir('checkpoints'))
print('data dirs present.')
print('checkpoints present:', sorted(ck))

## 4. Shared imports

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import roc_curve
from torch.utils.data import DataLoader
sns.set_theme(style='white')

from src import config
from src.config import TrainConfig
from src.data import manifests
from src.data.dataset import EmotionDataset, PairDataset, collate_emotion, collate_pair
from src.data.splits import partition_by_actor
from src.train_utils import set_seed, stratified_split
from src.models.speech_text import SpeechTextModule
from src.models.fv_litenet import FVLiteNet
from src.models.acenet import ACENet
from src.eval_stage1 import _collect_preds, metrics as emo_metrics, confusion as emo_conf, per_class_prf
from src.eval_stage2 import build_balanced_tagged, collect as s2_collect, auc_score, bin_metrics, dataset_of

cfg=TrainConfig(); set_seed(cfg.seed)
DEV='cuda' if torch.cuda.is_available() else 'cpu'
OUT='/content/drive/MyDrive/acenet_results'; os.makedirs(OUT, exist_ok=True)
print('device', DEV, '| outputs ->', OUT)

## 5. Stage-1 emotion recognition (Tables 2 and 3)

Each Stage-1 model is evaluated on the test partition of its dataset. Reported:
overall Accuracy and Weighted-F1, a per-class precision/recall/F1 table, and a
row-normalized confusion matrix.

In [ ]:
def eval_stage1(branch, dataset):
    ckpt=config.CKPT_ROOT/f'stage1_{branch}_{dataset}.pt'
    if not ckpt.exists(): return None
    names=config.EMOTION_DATASETS[dataset]['emotions']; nc=len(names)
    samples=manifests.build_emotion_samples(dataset)
    if dataset=='crema':
        _,_,te=partition_by_actor(samples, lambda s:s.group_key,(0.8,0.1,0.1),cfg.seed)
    else:
        _,_,te=stratified_split(samples, lambda s:s.emotion,(0.8,0.1,0.1),cfg.seed)
    model=(SpeechTextModule(n_classes=nc) if branch=='speech_text' else FVLiteNet(n_classes=nc)).to(DEV)
    model.load_state_dict(torch.load(ckpt, map_location=DEV))
    dl=DataLoader(EmotionDataset(te),batch_size=32,shuffle=False,num_workers=2,collate_fn=collate_emotion)
    y,p=_collect_preds(model, dl, branch, DEV)
    acc,wf1=emo_metrics(y,p,nc)
    return dict(branch=branch,dataset=dataset,names=names,acc=acc*100,wf1=wf1,
                y=y,p=p,n=len(y),prf=per_class_prf(y,p,names))

S1={}
for br in ['speech_text','visual']:
    for ds in ['crema','meld']:
        r=eval_stage1(br,ds)
        if r: S1[(br,ds)]=r
print('evaluated:', list(S1.keys()))

### 5.1 Summary — Accuracy and Weighted-F1

Table 2 corresponds to the MDCNN (speech-text) rows; Table 3 to the FV-LiteNet
(visual) rows.

In [ ]:
label={'speech_text':'MDCNN (Table 2)','visual':'FV-LiteNet (Table 3)'}
rows=[{'Model':label[br],'Dataset':ds.upper(),'ACC %':round(r['acc'],2),
       'Weighted-F1':round(r['wf1'],3),'n_test':r['n']} for (br,ds),r in S1.items()]
df_s1=pd.DataFrame(rows).sort_values(['Model','Dataset']).reset_index(drop=True)
df_s1.to_csv(f'{OUT}/stage1_summary.csv', index=False)
df_s1

In [ ]:
if not df_s1.empty:
    fig,ax=plt.subplots(figsize=(8,4.5)); x=np.arange(len(df_s1)); w=0.38
    ax.bar(x-w/2, df_s1['ACC %'], w, label='Accuracy %', color='#4C72B0')
    ax.bar(x+w/2, df_s1['Weighted-F1']*100, w, label='Weighted-F1 (×100)', color='#DD8452')
    ax.set_xticks(x); ax.set_xticklabels([f"{r.Dataset}\n{r.Model.split()[0]}" for r in df_s1.itertuples()], fontsize=8)
    ax.set_ylabel('score'); ax.set_ylim(0,100); ax.legend()
    ax.set_title('Stage-1 emotion recognition')
    for i,(a,f) in enumerate(zip(df_s1['ACC %'],df_s1['Weighted-F1']*100)):
        ax.text(i-w/2,a+1,f'{a:.1f}',ha='center',fontsize=7); ax.text(i+w/2,f+1,f'{f:.1f}',ha='center',fontsize=7)
    plt.tight_layout(); plt.savefig(f'{OUT}/stage1_summary.png',dpi=150,bbox_inches='tight'); plt.show()

### 5.2 Per-class precision / recall / F1

In [ ]:
for (br,ds),r in S1.items():
    print(f"\n[{label[br]} — {ds.upper()}]  ACC {r['acc']:.2f}%  Weighted-F1 {r['wf1']:.3f}  (n={r['n']})")
    t=pd.DataFrame(r['prf'], columns=['class','Precision','Recall','F1','support'])
    t[['Precision','Recall','F1']]=t[['Precision','Recall','F1']].round(3)
    t.to_csv(f"{OUT}/stage1_perclass_{br}_{ds}.csv", index=False)
    display(t)

### 5.3 Confusion matrices (row-normalized)

Corresponds to paper Figures 5 (speech-text) and 6 (visual).

In [ ]:
for (br,ds),r in S1.items():
    names=r['names']; cm=emo_conf(r['y'],r['p'],len(names)).astype(float)
    cmn=cm/cm.sum(1,keepdims=True).clip(min=1)
    fig,ax=plt.subplots(figsize=(1+0.85*len(names),0.85*len(names)))
    sns.heatmap(cmn,annot=True,fmt='.2f',cmap='Blues',xticklabels=names,yticklabels=names,
                ax=ax,vmin=0,vmax=1,cbar=True)
    ax.set_title(f"{label[br].split()[0]} — {ds.upper()} (ACC {r['acc']:.1f}%)")
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    plt.tight_layout(); plt.savefig(f'{OUT}/confusion_{br}_{ds}.png',dpi=150,bbox_inches='tight'); plt.show()

## 6. Stage-2 forgery detection (Table 4)

The full ACE-Net (frozen extractors + fusion + MLP) is evaluated on the
actor-disjoint test partition. Per pairing type — Genuine Pairs, Emotion
Tampering (P1), Cross-Identity Spliced (P2) — and overall. Forgery types are
scored one-vs-genuine.

In [ ]:
samples=build_balanced_tagged(cfg.seed)
_,_,te=partition_by_actor(samples, lambda s:s.group_key,(0.8,0.1,0.1),cfg.seed)
model=ACENet().to(DEV)
model.load_state_dict(torch.load(config.CKPT_ROOT/'stage2_acenet.pt', map_location=DEV))
model.eval()
y,p=s2_collect(model, te, DEV, 32, 2)
types=np.array([s.ptype for s in te]); dsets=np.array([dataset_of(s) for s in te])
gen_mask=(y==0)
print('test pairs:', len(te), '| genuine', int(gen_mask.sum()), '| fake', int((~gen_mask).sum()))

### 6.1 Table 4 — detection by pairing type

In [ ]:
rows=[]
for ds in sorted(set(dsets.tolist())):
    ds_gen=(dsets==ds)&gen_mask
    for pt in ['Genuine Pairs','Emotion Tampering','Cross-Identity Spliced']:
        sel=(dsets==ds)&(types==pt)
        if sel.sum()==0: continue
        if pt=='Genuine Pairs':
            acc=float((p[sel]<0.5).mean())
            rows.append({'Dataset':ds,'Pairing Type':pt,'ACC %':round(acc*100,1),
                         'Precision':None,'Recall':None,'F1':None,'AUC':None})
        else:
            idx=sel|ds_gen
            acc,pr,rc,f1=bin_metrics(y[idx],p[idx])
            rows.append({'Dataset':ds,'Pairing Type':pt,'ACC %':round(acc*100,1),
                         'Precision':round(pr,3),'Recall':round(rc,3),'F1':round(f1,3),
                         'AUC':round(auc_score(y[idx],p[idx]),3)})
oacc,op,orr,of1=bin_metrics(y,p)
rows.append({'Dataset':'OVERALL','Pairing Type':'all pairs','ACC %':round(oacc*100,1),
             'Precision':round(op,3),'Recall':round(orr,3),'F1':round(of1,3),'AUC':round(auc_score(y,p),3)})
df_s2=pd.DataFrame(rows); df_s2.to_csv(f'{OUT}/stage2_table4.csv', index=False)
df_s2

### 6.2 AUC by forgery type

In [ ]:
d=df_s2.dropna(subset=['AUC']).copy()
fig,ax=plt.subplots(figsize=(7,4.5)); x=np.arange(len(d))
ax.bar(x, d['AUC'], 0.5, color=['#55A868' if 'Cross' in t else '#C44E52' if 'Emotion' in t else '#4C72B0' for t in d['Pairing Type']])
ax.axhline(0.5, ls='--', c='gray', lw=1, label='chance')
ax.set_xticks(x); ax.set_xticklabels([t.replace(' ','\n') for t in d['Pairing Type']], fontsize=8)
ax.set_ylim(0,1.0); ax.set_ylabel('AUC'); ax.set_title('Stage-2 AUC by pairing type'); ax.legend()
for i,v in enumerate(d['AUC']): ax.text(i,v+0.01,f'{v:.3f}',ha='center',fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/stage2_auc_by_type.png',dpi=150,bbox_inches='tight'); plt.show()

### 6.3 ROC curves (per forgery type, one-vs-genuine)

In [ ]:
fig,ax=plt.subplots(figsize=(5.5,5.5))
for pt,c in [('Emotion Tampering','#C44E52'),('Cross-Identity Spliced','#55A868')]:
    sel=(types==pt); idx=sel|gen_mask
    fpr,tpr,_=roc_curve(y[idx], p[idx])
    ax.plot(fpr,tpr,color=c,label=f'{pt} (AUC {auc_score(y[idx],p[idx]):.3f})')
fpr,tpr,_=roc_curve(y,p)
ax.plot(fpr,tpr,color='#4C72B0',lw=2,label=f'Overall (AUC {auc_score(y,p):.3f})')
ax.plot([0,1],[0,1],'--',c='gray',lw=1)
ax.set_xlabel('false positive rate'); ax.set_ylabel('true positive rate')
ax.set_title('Stage-2 ROC'); ax.legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/stage2_roc.png',dpi=150,bbox_inches='tight'); plt.show()

### 6.4 Confusion by pairing type (threshold 0.5)

Rows are true pairing type; columns are the binary prediction.

In [ ]:
yhat=(p>=0.5).astype(int)
order=['Genuine Pairs','Emotion Tampering','Cross-Identity Spliced']
cm=np.zeros((3,2),dtype=int)
for i,pt in enumerate(order):
    sel=(types==pt)
    cm[i,0]=int((yhat[sel]==0).sum()); cm[i,1]=int((yhat[sel]==1).sum())
cmn=cm/cm.sum(1,keepdims=True).clip(min=1)
fig,ax=plt.subplots(figsize=(5,4))
sns.heatmap(cmn,annot=cm,fmt='d',cmap='Blues',xticklabels=['pred genuine','pred fake'],
            yticklabels=['Genuine','Emotion-Tamper','Cross-Identity'],ax=ax,vmin=0,vmax=1,cbar=True)
ax.set_title(f'Stage-2 confusion by type (overall AUC {auc_score(y,p):.3f})')
ax.set_xlabel('prediction'); ax.set_ylabel('true pairing type')
plt.tight_layout(); plt.savefig(f'{OUT}/stage2_confusion_by_type.png',dpi=150,bbox_inches='tight'); plt.show()

## 7. Outputs

`MyDrive/acenet_results/` contains:
- `stage1_summary.csv`, `stage1_summary.png`
- `stage1_perclass_{branch}_{dataset}.csv`
- `confusion_{branch}_{dataset}.png` (Figures 5/6)
- `stage2_table4.csv`
- `stage2_auc_by_type.png`, `stage2_roc.png`, `stage2_confusion_by_type.png`